In [1]:
import numpy as np 
import torch 
import pandas as pd
import os

# absolute path to RTSGAN repo
RTSGAN_PATH = "/path/to/Documents/RTSGAN"
import sys
sys.path.append(RTSGAN_PATH)
sys.path.append(RTSGAN_PATH + "/general")

import pickle
from general.missingprocessor import Processor
from fastNLP import DataSet, DataSetIter

/path/to/miniforge3/envs/rtsgan_env/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_data_OU = '/path/to/HyperNSDE/datasets/Simu_OU/simulated_dataset_3'
t_visits = 200
n_long_var = 3

In [3]:
def read_csv_values(path, to_torch=False, sep=',', header=None, index_col=None, engine='python', method='pandas'):
    """
    Read a CSV file and return its values as a numpy array or a torch tensor.
    
    Parameters
    ----------
        path : str
            Path to the CSV file.
        to_torch : bool, optional
            If True, convert the numpy array to a torch tensor. Default is False.
        sep : str, optional
            Delimiter to use. Default is ','.
        header : int, list of int, None, optional
            Row(s) to use as the column names. Default is None.
        index_col : int, str, sequence, or False, optional
            Column(s) to set as index. Default is None. 
        engine : str, optional
            Parser engine to use. Default is 'python'.
    Returns
    -------
        data : numpy.ndarray or torch.Tensor
            The values from the CSV file as a numpy array or torch tensor.
    """
    if method == 'pandas':
        data = pd.read_csv(path, sep=sep, header=header, index_col=index_col, engine=engine)
        data = data.values
    elif method == 'numpy':
        # Step 1: Load with NumPy as float32
        data = np.loadtxt(path, delimiter=sep, dtype=np.float32)

    if to_torch:
        data = torch.from_numpy(data).float()
    return data


def weighter(data):
    """
    Create a weight matrix and a values matrix from the input data.

    Parameters
    ----------
        data : torch.Tensor
            The input data tensor with shape (n_samples, n_timepoints, n_features).
    Returns
    -------
        values_matrix : torch.Tensor
            The values matrix with the same shape as the input data, where NaN values are replaced with 0.0.
        weight_matrix : torch.Tensor
            The weight matrix with the same shape as the input data, where NaN values are replaced with 0.0 and non-NaN values are replaced with 1.0.
    """
    mask = torch.isnan(data)
    weight_matrix = (~mask).float()
    values_matrix = torch.where(mask, torch.tensor(0.0), data)
    return values_matrix, weight_matrix

In [4]:
data = read_csv_values(os.path.join(path_data_OU, 'data_long.csv'), header=None, index_col=None, method='numpy')

n_patients = len(data)
data = torch.from_numpy(data).view(n_patients, t_visits, n_long_var + 1).float()

x, mask = weighter(data[:,:,1:])
time_grid = data[0, :, 0]
# t_over = (time_grid - time_grid[0]) / (time_grid[-1] - time_grid[0])

static_vals = read_csv_values(os.path.join(path_data_OU, 'data_static.csv'), to_torch=True)
static_types = read_csv_values(os.path.join(path_data_OU, 'data_static_types.csv'), header=None)
static_missing = read_csv_values(os.path.join(path_data_OU, 'data_static_missing.csv'), to_torch=True)

In [5]:
for i in range(x.shape[0]):
    observed_positions = torch.nonzero(mask[i] == 1, as_tuple=False)
    idx = observed_positions[np.random.randint(len(observed_positions))] 
    t, v = idx.tolist()
    data[i, t, v+1] = float('nan')
    

In [6]:
N, T, V = data[:,:,1:].shape

# Time vector
time = torch.tensor(time_grid).float()

# Flatten:
# PATIENT_ID: repeat each patient ID T times
patient_ids = torch.arange(N).repeat_interleave(T)

# Time: repeat time vector for each patient
times = time.repeat(N)

# Flatten the values: (N*T, V)
values = data[:,:,1:].reshape(N*T, V)

# Build column names
var_cols = [f"long_var_{i+1}" for i in range(V)]

# Create dataframe
df_long = pd.DataFrame(values.numpy(), columns=var_cols)
df_long.insert(0, "time", times.numpy())
df_long.insert(0, "PATIENT_ID", patient_ids.numpy())

df_long.head()

/path/to/miniforge3/envs/rtsgan_env/lib/python3.7/site-packages/ipykernel_launcher.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  after removing the cwd from sys.path.


,PATIENT_ID,time,long_var_1,long_var_2,long_var_3
0,0,0.000000,NaN,NaN,NaN
1,0,0.035176,NaN,NaN,NaN
2,0,0.070352,NaN,NaN,NaN
3,0,0.105528,NaN,NaN,NaN
4,0,0.140704,NaN,NaN,NaN


In [7]:
static_vals.shape

torch.Size([1000, 2])

In [8]:
N, V = static_vals.shape
var_cols_stat = [f"stat_var_{i+1}" for i in range(V)]
df_static = pd.DataFrame(static_vals.numpy(), columns=var_cols_stat)
df_static.head()

,stat_var_1,stat_var_2
0,0.033265,1.0
1,-0.034952,1.0
2,0.169440,1.0
3,0.027754,1.0
4,-0.141725,1.0


- refaire des dataframe à partir de ça 
- voir s'il faut normalizer les longitudinales

In [9]:
# Store for later metadata
static_variables = df_static.columns.tolist()
sta = df_static

In [10]:
# # problem with PATIENT_ID
# dyn = []
# for pid, subdf in df_long.groupby("PATIENT_ID"):
#     d = subdf.sort_values("Time").reset_index(drop=True)
#     d = d.drop(columns=["PATIENT_ID"])   # keep Time + variables
#     dyn.append(d)

# dynamic_variables = [c for c in dyn[0].columns if c != "Time"]

In [11]:
dyn = []

for pid, subdf in df_long.groupby("PATIENT_ID"):
    d = subdf.sort_values("time").reset_index(drop=True)
    d = d.drop(columns=["PATIENT_ID"])   # keep Time + vars
    
    # Keep only rows where at least one dynamic variable is non-NaN
    # Excluding "Time" column
    mask_valid = d.drop(columns=["time"]).notna().any(axis=1)
    d = d[mask_valid].reset_index(drop=True)

    dyn.append(d)

# Variables (same logic)
dynamic_variables = [c for c in dyn[0].columns if c != "time"]

In [12]:
print(static_variables)
print(dynamic_variables)

['stat_var_1', 'stat_var_2']
['long_var_1', 'long_var_2', 'long_var_3']


In [13]:
val_ratio = 0.25
n = len(sta)
idx = np.arange(n)
rng = np.random.RandomState(0)
rng.shuffle(idx)

n_val = int(n * val_ratio)

val_idx = idx[:n_val]
train_idx = idx[n_val:]

# Split static
sta_train = sta.iloc[train_idx].reset_index(drop=True)
sta_val   = sta.iloc[val_idx].reset_index(drop=True)

# Split dynamics
dyn_train = [dyn[i] for i in train_idx]
dyn_val   = [dyn[i] for i in val_idx]

In [14]:
# Static variable metadata
sta_types = ["continuous", "binary", "int"]

# Dynamic variables metadata
dyn_types = ['continuous']*len(dyn[0].columns)
print(dyn_types)

['continuous', 'continuous', 'continuous', 'continuous']


In [15]:
train_seq_len = [len(x) for x in dyn_train]
sta_train["seq_len"]=np.array(train_seq_len)

d_P = Processor(dyn_types, use_pri='time')
s_P = Processor(sta_types)

dynamics = pd.concat(dyn_train)
s_P.fit(sta_train)
d_P.fit(dynamics)

stat_var_1 1 None continuous None
[0.81119704] [-0.99805021]
stat_var_2 1 None binary None
seq_len 1 None int None
[67.] [30.]
time 1 None continuous None
[7.] [0.]
long_var_1 1 0.00748220694689458 continuous None
[1.346012] [-0.6369309]
long_var_2 1 0.007664699799257863 continuous None
[4.4179983] [-0.7836022]
long_var_3 1 0.007664699799257863 continuous None
[2.1231542] [-0.6729889]


In [16]:
dynamics[48:]

,time,long_var_1,long_var_2,long_var_3
48,6.613065,0.361968,0.227010,-0.213354
49,6.788945,0.391074,0.283086,-0.165999
0,0.000000,-0.069977,-0.012028,-0.026334
1,0.140704,-0.031760,0.067918,0.079854
2,0.175879,-0.016530,0.023660,0.057493
...,...,...,...,...
47,6.507538,0.609570,-0.118863,0.538425
48,6.648241,0.659726,-0.139157,0.556881
49,6.753769,0.739626,-0.152565,0.570753
50,6.788945,0.724806,-0.142375,0.591283


In [17]:
dyn[0].head()

,time,long_var_1,long_var_2,long_var_3
0,0.633166,-0.012220,-0.010618,-0.097512
1,0.773869,0.044480,-0.078603,-0.113638
2,0.844221,0.076882,-0.093764,-0.093326
3,1.125628,0.245131,-0.083088,-0.044870
4,1.160804,0.289902,-0.108586,-0.048881


In [18]:
sta_train.head()

,stat_var_1,stat_var_2,seq_len
0,-0.243841,0.0,50
1,0.307195,1.0,38
2,0.166791,0.0,42
3,-0.195964,1.0,49
4,-0.349353,0.0,34


In [19]:
dyn[1]

,time,long_var_1,long_var_2,long_var_3
0,0.000000,-0.055102,-0.003811,-0.044831
1,0.035176,-0.028619,-0.027347,-0.018492
2,0.070352,-0.015209,-0.034846,-0.006969
3,0.211055,0.006738,-0.076318,0.089201
4,0.386935,-0.014872,-0.151724,0.017282
5,0.422111,-0.053273,NaN,0.057252
6,0.633166,-0.046285,-0.099407,0.123790
7,0.668342,-0.020962,-0.118649,0.144562
8,0.703518,-0.036255,-0.117029,0.150712
9,1.266332,0.017339,-0.045777,0.129027


In [20]:
def build_dataset(sta, dyn, seq_len):
    s = s_P.transform(sta)
    
    d_lis=[d_P.transform(ds) for ds in dyn]
    d = [x[0].tolist() for x in d_lis]
    lag = [x[1].tolist() for x in d_lis]
    mask = [x[2].tolist() for x in d_lis]
    times = [x[-1].tolist() for x in d_lis]
    priv = [x[3].tolist() for x in d_lis]
    nex = [x[4].tolist() for x in d_lis]
    # label = [float(x[-2]) for x in s] 
    label = [0.0] * len(s)
    print(len(d))
    print(len(d[0]))
    dataset = DataSet({"seq_len": seq_len, 
                       "dyn": d, "lag":lag, "mask": mask,
                       "sta": s, "times":times, "priv":priv, "nex":nex, "label": label
                      })
    return dataset

In [21]:
train_dataset = build_dataset(sta_train, dyn_train, train_seq_len)

750
50


In [22]:
dyn[0]

,time,long_var_1,long_var_2,long_var_3
0,0.633166,-0.012220,-0.010618,-0.097512
1,0.773869,0.044480,-0.078603,-0.113638
2,0.844221,0.076882,-0.093764,-0.093326
3,1.125628,0.245131,-0.083088,-0.044870
4,1.160804,0.289902,-0.108586,-0.048881
5,1.407035,0.271882,-0.180680,-0.090528
6,1.442211,0.297770,-0.190321,-0.055821
7,1.512563,0.253946,-0.159247,-0.093575
8,1.582915,0.301769,-0.139533,-0.160417
9,1.899498,0.450439,-0.161098,-0.046463


In [24]:
train_dataset

+---------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------+
| seq_len | dyn          | lag          | mask         | sta          | times        | priv         | nex          | label |
+---------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------+
| 50      | [[0.31251... | [[0.00502... | [[1.0, 1.... | [0.416863... | [[0.00502... | [[0.0, 0.... | [[0.0, 0.... | 0.0   |
| 38      | [[0.28591... | [[0.0, 0.... | [[1.0, 1.... | [0.721430... | [[0.0], [... | [[0.0, 0.... | [[0.0, 0.... | 0.0   |
| 42      | [[0.30598... | [[0.03015... | [[1.0, 1.... | [0.643826... | [[0.03015... | [[0.0, 0.... | [[0.0, 0.... | 0.0   |
| 49      | [[0.28807... | [[0.00502... | [[1.0, 1.... | [0.443325... | [[0.00502... | [[0.0, 0.... | [[0.0, 0.... | 0.0   |
| 34      | [[0.30198... | [[0.05527... | [[1.0, 1.... | [0.358545... | [[0.05527... | [[0.0, 0.... | [[0.0, 0.... | 0.0   |


In [25]:
val_seq_len = [len(x) for x in dyn_val]
sta_val["seq_len"]=np.array(val_seq_len)
val_set = build_dataset(sta_val, dyn_val, val_seq_len)

250
44


In [26]:
finaldic = {
    "train_set": train_dataset,
    'raw_set': (sta_train,dyn_train),
    'test_set': (sta_val,dyn_val),
    'val_set': val_set,
    "dynamic_processor": d_P,
    "static_processor":s_P
}

with open("OU_v2_rtsgan.pkl", "wb") as f:
    pickle.dump(finaldic, f)

print("Saved RTSGAN-ready dataset: OU_v2_rtsgan.pkl")

Saved RTSGAN-ready dataset: OU_v2_rtsgan.pkl
